# BigCloneBench Type 1 Tuning Analysis

Reads the outputs produced by `04_tune_pss_wasserstein.py`: trained threshold JSON and pair-level score CSV.


In [1]:
from pathlib import Path
import html
import json
import pickle
import sys

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from IPython.display import HTML, display


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "spectral_code").exists() and (candidate / "pipelines").exists():
            return candidate
    raise RuntimeError("Project root not found.")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from spectral_code.utils.dataset_paths import bcb_type_dir, output_root_for

CLONE_TYPE = 1
DATA_DIR = bcb_type_dir(CLONE_TYPE)
NON_CLONE_DATA_DIR = bcb_type_dir("non_clone")
NON_CLONE_OUTPUT_ROOT = output_root_for("bcb", "non_clone")
OUTPUT_ROOT = output_root_for("bcb", CLONE_TYPE)
TUNING_JSON = OUTPUT_ROOT / "trained_bcb_type1_f1_pss_wasserstein.json"
PAIR_SCORES_CSV = OUTPUT_ROOT / "pair_scores_trained_bcb_type1_f1_pss_wasserstein.csv"
GRAPH_MANIFEST_PATH = OUTPUT_ROOT / "clean_graphs" / "graph_shards_manifest.json"
SPECTRAL_MANIFEST_PATH = OUTPUT_ROOT / "spectral_features" / "spectral_features_manifest.json"
NON_CLONE_GRAPH_MANIFEST_PATH = NON_CLONE_OUTPUT_ROOT / "clean_graphs" / "graph_shards_manifest.json"
NON_CLONE_SPECTRAL_MANIFEST_PATH = NON_CLONE_OUTPUT_ROOT / "spectral_features" / "spectral_features_manifest.json"
PIPELINE_TIMINGS_PATH = OUTPUT_ROOT / "pipeline_timings.json"

for required in [TUNING_JSON, PAIR_SCORES_CSV, GRAPH_MANIFEST_PATH, SPECTRAL_MANIFEST_PATH]:
    if not required.exists():
        raise FileNotFoundError(f"Required artifact not found: {required}")

trained = json.loads(TUNING_JSON.read_text(encoding="utf-8"))
trained_df = pd.DataFrame(trained)
if trained_df.empty:
    raise RuntimeError(f"No tuning rows found in {TUNING_JSON}")

if "decision_threshold" not in trained_df.columns and "best_threshold" in trained_df.columns:
    trained_df["decision_threshold"] = trained_df["best_threshold"].astype(float).map(lambda value: float(np.nextafter(value, -np.inf)))
    display(HTML("<b>Note:</b> this tuning JSON has no recorded <code>decision_threshold</code>. Re-run <code>04_tune_pss_wasserstein.py</code> to refresh the stored metrics with the current threshold rule."))

sort_cols = [col for col in ["train_f1", "train_accuracy", "best_metric"] if col in trained_df.columns]
trained_df = trained_df.sort_values(sort_cols, ascending=[False] * len(sort_cols)).reset_index(drop=True)
example_candidates = trained_df.copy()
if {"train_precision", "train_recall"}.issubset(example_candidates.columns):
    mixed = example_candidates[(example_candidates["train_precision"] < 1.0) & (example_candidates["train_recall"] < 1.0)]
    if not mixed.empty:
        example_candidates = mixed
best_config = example_candidates.iloc[0].to_dict()
ACTIVE_GRAPH = str(best_config["graph_type"])
ACTIVE_METRIC = str(best_config["metric"])
ACTIVE_K = "full" if pd.isna(best_config.get("k_eigen")) else str(best_config.get("k_eigen"))
ACTIVE_THRESHOLD = float(best_config["best_threshold"])
METRICS = sorted(trained_df["metric"].dropna().astype(str).unique())
CANONICAL_GRAPH_ORDER = ["ast", "cfg", "ddg", "pdg", "cpg"]
available_graphs = set(trained_df["graph_type"].dropna().astype(str).unique())
GRAPH_TYPES = [graph for graph in CANONICAL_GRAPH_ORDER if graph in available_graphs]
GRAPH_TYPES.extend(sorted(available_graphs - set(GRAPH_TYPES)))

print("Output root:", OUTPUT_ROOT)
print("Active config for TP/FP/FN/TN examples:", ACTIVE_GRAPH, ACTIVE_K, ACTIVE_METRIC, "threshold", ACTIVE_THRESHOLD)
display(trained_df)


FileNotFoundError: Required artifact not found: C:\Users\koush\PyProjects\spectrals\outputs\bcb\type1\trained_bcb_type1_f1_pss_wasserstein.json

In [ ]:
def pair_key(left_id, right_id) -> tuple[str, str]:
    return str(left_id), str(right_id)


PERFECT_METRIC_EPSILON = 1e-12


def decision_threshold(threshold: float) -> float:
    return float(np.nextafter(float(threshold), -np.inf))


def prediction_label(score: float, threshold: float) -> int:
    return int(float(score) >= decision_threshold(threshold))


def confusion_name(label: int, pred: int) -> str:
    if label == 1 and pred == 1:
        return "TP"
    if label == 0 and pred == 1:
        return "FP"
    if label == 1 and pred == 0:
        return "FN"
    return "TN"


def select_confusion_examples(csv_path: Path, graph_type: str, metric: str, k_label: str, threshold: float) -> dict[str, dict]:
    selected = {}
    usecols = ["left_id", "right_id", "label", "graph_type", "k_eigen", "metric", "score"]
    for chunk in pd.read_csv(csv_path, usecols=usecols, dtype={"left_id": str, "right_id": str, "k_eigen": str}, chunksize=250_000):
        chunk = chunk[
            (chunk["graph_type"].astype(str) == graph_type)
            & (chunk["metric"].astype(str) == metric)
            & (chunk["k_eigen"].astype(str) == k_label)
        ]
        if chunk.empty:
            continue
        chunk["pred"] = (chunk["score"].astype(float) >= decision_threshold(threshold)).astype(int)
        for row in chunk.itertuples(index=False):
            name = confusion_name(int(row.label), int(row.pred))
            if name not in selected:
                selected[name] = {
                    "left_id": str(row.left_id),
                    "right_id": str(row.right_id),
                    "label": int(row.label),
                    "pred": int(row.pred),
                    "active_score": float(row.score),
                }
            if {"TP", "FP", "FN", "TN"}.issubset(selected):
                return selected
    return selected


examples = select_confusion_examples(PAIR_SCORES_CSV, ACTIVE_GRAPH, ACTIVE_METRIC, ACTIVE_K, ACTIVE_THRESHOLD)
print("Selected example types:", sorted(examples))
display(pd.DataFrame.from_dict(examples, orient="index"))

example_keys = {pair_key(item["left_id"], item["right_id"]) for item in examples.values()}
needed_ids = {method_id for key in example_keys for method_id in key}

# Collect all metric scores for the selected pairs across every graph on the active k.
score_rows = []
usecols = ["left_id", "right_id", "label", "graph_type", "k_eigen", "metric", "score"]
for chunk in pd.read_csv(PAIR_SCORES_CSV, usecols=usecols, dtype={"left_id": str, "right_id": str, "k_eigen": str}, chunksize=250_000):
    chunk = chunk[chunk["k_eigen"].astype(str) == ACTIVE_K]
    if chunk.empty:
        continue
    mask = [(left, right) in example_keys for left, right in zip(chunk["left_id"], chunk["right_id"])]
    if any(mask):
        score_rows.append(chunk.loc[mask].copy())
metric_scores_df = pd.concat(score_rows, ignore_index=True) if score_rows else pd.DataFrame(columns=usecols)
display(metric_scores_df.sort_values(["left_id", "right_id", "graph_type", "metric"]))


In [ ]:
def load_code_for_ids(path: Path, wanted_ids: set[str]) -> dict[str, str]:
    code_map = {}
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            function_id = str(row["idx"])
            if function_id in wanted_ids:
                code_map[function_id] = row["func"]
                if len(code_map) == len(wanted_ids):
                    break
    return code_map


def load_graphs_for_ids(manifest_path: Path, wanted_ids: set[str]) -> dict[str, dict[str, nx.DiGraph]]:
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    graphs = {}
    for shard_path in manifest.get("shards", []):
        with Path(shard_path).open("rb") as f:
            shard = pickle.load(f)
        for method_id in list(wanted_ids - set(graphs)):
            if method_id in shard:
                graphs[method_id] = shard[method_id]
        if len(graphs) == len(wanted_ids):
            break
    return graphs


def load_spectral_features_for_ids(manifest_path: Path, wanted_ids: set[str]) -> dict[str, dict]:
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    features = {}
    for shard_path in manifest.get("shards", []):
        with Path(shard_path).open("rb") as f:
            shard = pickle.load(f)
        for method_id in list(wanted_ids - set(features)):
            if method_id in shard:
                features[method_id] = shard[method_id]
        if len(features) == len(wanted_ids):
            break
    return features


code_map = load_code_for_ids(DATA_DIR / "data.jsonl", needed_ids)
missing_code_ids = needed_ids - set(code_map)
if missing_code_ids and (NON_CLONE_DATA_DIR / "data.jsonl").exists():
    code_map.update(load_code_for_ids(NON_CLONE_DATA_DIR / "data.jsonl", missing_code_ids))
graph_map = load_graphs_for_ids(GRAPH_MANIFEST_PATH, needed_ids)
missing_graph_ids = needed_ids - set(graph_map)
if missing_graph_ids and NON_CLONE_GRAPH_MANIFEST_PATH.exists():
    graph_map.update(load_graphs_for_ids(NON_CLONE_GRAPH_MANIFEST_PATH, missing_graph_ids))
spectral_map = load_spectral_features_for_ids(SPECTRAL_MANIFEST_PATH, needed_ids)
missing_spectral_ids = needed_ids - set(spectral_map)
if missing_spectral_ids and NON_CLONE_SPECTRAL_MANIFEST_PATH.exists():
    spectral_map.update(load_spectral_features_for_ids(NON_CLONE_SPECTRAL_MANIFEST_PATH, missing_spectral_ids))

print("Loaded code snippets:", len(code_map), "missing:", len(needed_ids - set(code_map)))
print("Loaded graph records:", len(graph_map), "missing:", len(needed_ids - set(graph_map)))
print("Loaded spectral records:", len(spectral_map), "missing:", len(needed_ids - set(spectral_map)))


def eigenvalues_for(method_id: str, graph_type: str) -> np.ndarray:
    values = spectral_map.get(str(method_id), {}).get(graph_type, {}).get("eigenvalues", [])
    values = np.asarray(values, dtype=np.float64)
    return values[np.isfinite(values)]


def draw_graph(graph: nx.DiGraph | None, title: str, ax) -> None:
    ax.set_title(title, fontsize=9)
    ax.axis("off")
    if graph is None or graph.number_of_nodes() == 0:
        ax.text(0.5, 0.5, "empty", ha="center", va="center")
        return
    graph = nx.DiGraph(graph)
    shown_graph = graph
    if shown_graph.number_of_nodes() > 100:
        shown_graph = shown_graph.subgraph(list(shown_graph.nodes())[:100]).copy()
    pos = nx.spring_layout(shown_graph, seed=42)
    nx.draw_networkx_edges(shown_graph, pos, ax=ax, arrows=False, alpha=0.25, width=0.8)
    nx.draw_networkx_nodes(shown_graph, pos, ax=ax, node_size=20, alpha=0.85)
    ax.text(
        0.01,
        0.02,
        f"{graph.number_of_nodes()} nodes / {graph.number_of_edges()} edges",
        transform=ax.transAxes,
        fontsize=7,
        color="#555",
    )


def show_code_pair(left_id: str, right_id: str, title: str) -> None:
    left_code = html.escape(code_map.get(str(left_id), ""))
    right_code = html.escape(code_map.get(str(right_id), ""))
    display(HTML(f"""
    <h3>{html.escape(title)}</h3>
    <div style="display:grid;grid-template-columns:1fr 1fr;gap:12px;align-items:start;">
      <div><b>left id: {html.escape(str(left_id))}</b><pre style="white-space:pre-wrap;font-size:12px;line-height:1.35;border:1px solid #d0d7de;padding:10px;border-radius:6px;max-height:420px;overflow:auto;">{left_code}</pre></div>
      <div><b>right id: {html.escape(str(right_id))}</b><pre style="white-space:pre-wrap;font-size:12px;line-height:1.35;border:1px solid #d0d7de;padding:10px;border-radius:6px;max-height:420px;overflow:auto;">{right_code}</pre></div>
    </div>
    """))


def show_pair_case(case_name: str, item: dict) -> None:
    left_id, right_id = item["left_id"], item["right_id"]
    title = f"{case_name}: label={item['label']} pred={item['pred']} selected by {ACTIVE_GRAPH}/{ACTIVE_METRIC} threshold={ACTIVE_THRESHOLD:.17g}"
    show_code_pair(left_id, right_id, title)

    pair_scores = metric_scores_df[(metric_scores_df["left_id"] == left_id) & (metric_scores_df["right_id"] == right_id)]
    if not pair_scores.empty:
        display_scores = pair_scores[["graph_type", "k_eigen", "metric", "score"]].sort_values(["graph_type", "metric"]).copy()
        display_scores["score"] = display_scores["score"].map(lambda value: f"{float(value):.17g}")
        display(display_scores)

    fig, axes = plt.subplots(len(GRAPH_TYPES), 4, figsize=(16, 3.0 * len(GRAPH_TYPES)))
    if len(GRAPH_TYPES) == 1:
        axes = np.asarray([axes])
    for row, graph_type in enumerate(GRAPH_TYPES):
        draw_graph(graph_map.get(left_id, {}).get(graph_type), f"left {graph_type.upper()} graph", axes[row][0])
        draw_graph(graph_map.get(right_id, {}).get(graph_type), f"right {graph_type.upper()} graph", axes[row][1])
        for ax, method_id, side in [(axes[row][2], left_id, "left"), (axes[row][3], right_id, "right")]:
            values = eigenvalues_for(method_id, graph_type)
            ax.set_title(f"{side} {graph_type.upper()} eigenvalues", fontsize=9)
            if values.size:
                ax.plot(np.arange(1, values.size + 1), np.sort(values), linewidth=1.1)
                ax.grid(alpha=0.2)
            else:
                ax.text(0.5, 0.5, "missing", ha="center", va="center", transform=ax.transAxes)
                ax.set_xticks([])
                ax.set_yticks([])
    plt.tight_layout()
    plt.show()


for case_name in ["TP", "FP", "FN", "TN"]:
    if case_name in examples:
        show_pair_case(case_name, examples[case_name])
    else:
        display(HTML(f"<b>{case_name}</b>: no example found for the active config."))


In [ ]:
def build_score_histograms(csv_path: Path, graph_types: list[str], metrics: list[str], k_label: str, bins: np.ndarray) -> dict:
    hist = {
        (graph, metric, label): np.zeros(len(bins) - 1, dtype=np.int64)
        for graph in graph_types
        for metric in metrics
        for label in [0, 1]
    }
    usecols = ["graph_type", "k_eigen", "metric", "label", "score"]
    for chunk in pd.read_csv(csv_path, usecols=usecols, dtype={"k_eigen": str}, chunksize=500_000):
        chunk = chunk[chunk["k_eigen"].astype(str) == k_label]
        if chunk.empty:
            continue
        chunk["label"] = chunk["label"].astype(int)
        chunk["score"] = chunk["score"].astype(float).clip(0, 1)
        for graph in graph_types:
            graph_chunk = chunk[chunk["graph_type"].astype(str) == graph]
            if graph_chunk.empty:
                continue
            for metric in metrics:
                metric_chunk = graph_chunk[graph_chunk["metric"].astype(str) == metric]
                if metric_chunk.empty:
                    continue
                for label in [0, 1]:
                    values = metric_chunk.loc[metric_chunk["label"] == label, "score"].to_numpy()
                    if values.size:
                        hist[(graph, metric, label)] += np.histogram(values, bins=bins)[0]
    return hist


def plot_metric_distribution(metric: str, graph_types: list[str], histograms: dict, bins: np.ndarray) -> None:
    centers = (bins[:-1] + bins[1:]) / 2
    width = float(bins[1] - bins[0])
    fig, axes = plt.subplots(
        len(graph_types),
        1,
        figsize=(11.5, max(3.0, 2.25 * len(graph_types))),
        sharex=True,
        squeeze=False,
    )
    colors = {0: "#4C78A8", 1: "#F58518"}
    labels = {0: "non-clone label 0", 1: f"type-{CLONE_TYPE} clone label 1"}

    for row, graph in enumerate(graph_types):
        ax = axes[row][0]
        max_y = 0.0
        for label in [0, 1]:
            counts = histograms[(graph, metric, label)]
            total = int(counts.sum())
            fraction = counts / total if total else counts.astype(float)
            max_y = max(max_y, float(fraction.max()) if fraction.size else 0.0)
            ax.bar(
                centers,
                fraction,
                width=width * 0.92,
                color=colors[label],
                alpha=0.58,
                edgecolor="white",
                linewidth=0.25,
                label=f"{labels[label]} (n={total:,})",
            )

        threshold_row = trained_df[
            (trained_df["graph_type"].astype(str) == graph)
            & (trained_df["metric"].astype(str) == metric)
            & (trained_df["k_eigen"].fillna("full").astype(str) == ACTIVE_K)
        ]
        if not threshold_row.empty:
            threshold = float(threshold_row.iloc[0]["best_threshold"])
            ax.axvline(threshold, color="#222222", linestyle="--", linewidth=1.1, alpha=0.8)
            ax.text(
                threshold,
                max_y * 0.92 if max_y else 0.02,
                f"  th={threshold:.3f}",
                rotation=90,
                va="top",
                ha="left",
                fontsize=8,
                color="#222222",
            )

        ax.set_title(f"{metric.upper()} distribution for {graph.upper()}", fontsize=11, pad=7)
        ax.set_ylabel("fraction of class", fontsize=9)
        ax.set_ylim(0, max(0.04, max_y * 1.18))
        ax.grid(True, axis="both", alpha=0.18, linewidth=0.8)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.legend(loc="upper left", fontsize=8, frameon=True)

    axes[-1][0].set_xlabel(f"{metric} similarity", fontsize=10)
    axes[-1][0].set_xlim(0, 1)
    fig.suptitle(f"Type-{CLONE_TYPE} pair score distributions by label - {metric.upper()}", y=0.995, fontsize=13)
    fig.tight_layout(rect=[0, 0, 1, 0.985])
    plt.show()


bins = np.linspace(0.0, 1.0, 71)
histograms = build_score_histograms(PAIR_SCORES_CSV, GRAPH_TYPES, METRICS, ACTIVE_K, bins)

for metric in METRICS:
    plot_metric_distribution(metric, GRAPH_TYPES, histograms, bins)


## Misclassified Pair Inspection

Finds pairs misclassified by every non-perfect graph/metric configuration and renders a few representative cases with code, graphs, eigenvalues, and scores.


In [ ]:
MAX_CONFIGS_TO_SCAN_FOR_ERRORS = 20
MAX_MISCLASSIFIED_CASES_TO_RENDER = 6


def k_label_for_value(value) -> str:
    return "full" if pd.isna(value) else str(value)


def config_display_name(row: pd.Series) -> str:
    return f"{row['graph_type']}/{row['metric']}/k={k_label_for_value(row.get('k_eigen'))}/th={float(row['best_threshold']):.17g}"


def collect_misclassified_pairs(csv_path: Path, configs: pd.DataFrame, max_cases_per_config: int = 3) -> tuple[pd.DataFrame, pd.DataFrame]:
    usecols = ["left_id", "right_id", "label", "graph_type", "k_eigen", "metric", "score"]
    summaries = []
    cases = []

    for _, config in configs.head(MAX_CONFIGS_TO_SCAN_FOR_ERRORS).iterrows():
        graph_type = str(config["graph_type"])
        metric = str(config["metric"])
        k_label = k_label_for_value(config.get("k_eigen"))
        threshold = float(config["best_threshold"])
        error_count = 0
        rendered_for_config = 0

        for chunk in pd.read_csv(csv_path, usecols=usecols, dtype={"left_id": str, "right_id": str, "k_eigen": str}, chunksize=250_000):
            chunk = chunk[
                (chunk["graph_type"].astype(str) == graph_type)
                & (chunk["metric"].astype(str) == metric)
                & (chunk["k_eigen"].astype(str) == k_label)
            ]
            if chunk.empty:
                continue

            scores = chunk["score"].astype(float)
            chunk = chunk.copy()
            chunk["pred"] = (scores >= decision_threshold(threshold)).astype(int)
            chunk["label"] = chunk["label"].astype(int)
            wrong = chunk[chunk["label"] != chunk["pred"]].copy()
            if wrong.empty:
                continue

            error_count += len(wrong)
            wrong["error_type"] = np.where(
                (wrong["label"] == 1) & (wrong["pred"] == 0),
                "FN",
                "FP",
            )
            wrong["active_graph_type"] = graph_type
            wrong["active_metric"] = metric
            wrong["active_k_eigen"] = k_label
            wrong["active_threshold"] = threshold
            wrong["active_decision_threshold"] = decision_threshold(threshold)
            wrong["score_minus_threshold"] = wrong["score"].astype(float) - threshold
            wrong["score_minus_decision_threshold"] = wrong["score"].astype(float) - decision_threshold(threshold)

            if rendered_for_config < max_cases_per_config:
                take = wrong.head(max_cases_per_config - rendered_for_config)
                cases.append(take)
                rendered_for_config += len(take)

        summaries.append({
            "graph_type": graph_type,
            "metric": metric,
            "k_eigen": k_label,
            "threshold": threshold,
            "decision_threshold": decision_threshold(threshold),
            "errors": int(error_count),
            "reported_train_accuracy": float(config.get("train_accuracy", np.nan)),
            "reported_train_precision": float(config.get("train_precision", np.nan)),
            "reported_train_recall": float(config.get("train_recall", np.nan)),
            "reported_train_f1": float(config.get("train_f1", np.nan)),
        })

    summary_df = pd.DataFrame(summaries).sort_values(["errors", "graph_type", "metric"], ascending=[False, True, True])
    cases_df = pd.concat(cases, ignore_index=True) if cases else pd.DataFrame(columns=usecols + [
        "pred", "error_type", "active_graph_type", "active_metric", "active_k_eigen", "active_threshold", "active_decision_threshold", "score_minus_threshold", "score_minus_decision_threshold"
    ])
    return summary_df, cases_df


nonperfect_configs = trained_df[
    (trained_df.get("train_accuracy", 1.0).astype(float) < 1.0 - PERFECT_METRIC_EPSILON)
    | (trained_df.get("train_precision", 1.0).astype(float) < 1.0 - PERFECT_METRIC_EPSILON)
    | (trained_df.get("train_recall", 1.0).astype(float) < 1.0 - PERFECT_METRIC_EPSILON)
].copy()

if nonperfect_configs.empty:
    display(HTML("<b>No misclassified pairs found in the recorded tuning summary.</b>"))
else:
    nonperfect_configs = nonperfect_configs.sort_values(["train_accuracy", "train_f1"], ascending=[True, True])
    mis_summary_df, mis_cases_df = collect_misclassified_pairs(PAIR_SCORES_CSV, nonperfect_configs)
    display(mis_summary_df)

    if mis_cases_df.empty:
        display(HTML("<b>No pair-level mistakes were found in the score CSV with the tolerant threshold comparison.</b>"))
    else:
        display_columns = [
            "error_type", "active_graph_type", "active_metric", "active_k_eigen",
            "left_id", "right_id", "label", "pred", "score", "active_threshold", "active_decision_threshold", "score_minus_threshold", "score_minus_decision_threshold",
        ]
        display_mis = mis_cases_df[display_columns].head(50).copy()
        for col in ["score", "active_threshold", "active_decision_threshold", "score_minus_threshold", "score_minus_decision_threshold"]:
            display_mis[col] = display_mis[col].map(lambda value: f"{float(value):.17g}")
        display(display_mis)

        render_cases_df = mis_cases_df.head(MAX_MISCLASSIFIED_CASES_TO_RENDER).copy()
        mis_keys = {pair_key(row.left_id, row.right_id) for row in render_cases_df.itertuples(index=False)}
        mis_needed_ids = {method_id for key in mis_keys for method_id in key}

        missing_code_ids = mis_needed_ids - set(code_map)
        missing_graph_ids = mis_needed_ids - set(graph_map)
        missing_spectral_ids = mis_needed_ids - set(spectral_map)
        if missing_code_ids:
            code_map.update(load_code_for_ids(DATA_DIR / "data.jsonl", missing_code_ids))
            missing_code_ids = mis_needed_ids - set(code_map)
            if missing_code_ids and (NON_CLONE_DATA_DIR / "data.jsonl").exists():
                code_map.update(load_code_for_ids(NON_CLONE_DATA_DIR / "data.jsonl", missing_code_ids))
        if missing_graph_ids:
            graph_map.update(load_graphs_for_ids(GRAPH_MANIFEST_PATH, missing_graph_ids))
            missing_graph_ids = mis_needed_ids - set(graph_map)
            if missing_graph_ids and NON_CLONE_GRAPH_MANIFEST_PATH.exists():
                graph_map.update(load_graphs_for_ids(NON_CLONE_GRAPH_MANIFEST_PATH, missing_graph_ids))
        if missing_spectral_ids:
            spectral_map.update(load_spectral_features_for_ids(SPECTRAL_MANIFEST_PATH, missing_spectral_ids))
            missing_spectral_ids = mis_needed_ids - set(spectral_map)
            if missing_spectral_ids and NON_CLONE_SPECTRAL_MANIFEST_PATH.exists():
                spectral_map.update(load_spectral_features_for_ids(NON_CLONE_SPECTRAL_MANIFEST_PATH, missing_spectral_ids))

        score_rows = []
        usecols = ["left_id", "right_id", "label", "graph_type", "k_eigen", "metric", "score"]
        for chunk in pd.read_csv(PAIR_SCORES_CSV, usecols=usecols, dtype={"left_id": str, "right_id": str, "k_eigen": str}, chunksize=250_000):
            mask = [(left, right) in mis_keys for left, right in zip(chunk["left_id"], chunk["right_id"])]
            if any(mask):
                score_rows.append(chunk.loc[mask].copy())
        mis_metric_scores_df = pd.concat(score_rows, ignore_index=True) if score_rows else pd.DataFrame(columns=usecols)

        def show_misclassified_case(row: pd.Series) -> None:
            left_id, right_id = str(row["left_id"]), str(row["right_id"])
            title = (
                f"{row['error_type']}: label={int(row['label'])} pred={int(row['pred'])} "
                f"by {row['active_graph_type']}/{row['active_metric']} "
                f"threshold={float(row['active_threshold']):.17g} "
                f"decision_threshold={float(row['active_decision_threshold']):.17g} "
                f"score={float(row['score']):.17g}"
            )
            show_code_pair(left_id, right_id, title)

            pair_scores = mis_metric_scores_df[(mis_metric_scores_df["left_id"] == left_id) & (mis_metric_scores_df["right_id"] == right_id)]
            if not pair_scores.empty:
                display_scores = pair_scores[["graph_type", "k_eigen", "metric", "score"]].sort_values(["graph_type", "metric"]).copy()
                display_scores["score"] = display_scores["score"].map(lambda value: f"{float(value):.17g}")
                display(display_scores)

            fig, axes = plt.subplots(len(GRAPH_TYPES), 4, figsize=(16, 3.0 * len(GRAPH_TYPES)))
            if len(GRAPH_TYPES) == 1:
                axes = np.asarray([axes])
            for graph_row, graph_type in enumerate(GRAPH_TYPES):
                highlight = graph_type == str(row["active_graph_type"])
                suffix = " *mistake config*" if highlight else ""
                draw_graph(graph_map.get(left_id, {}).get(graph_type), f"left {graph_type.upper()} graph{suffix}", axes[graph_row][0])
                draw_graph(graph_map.get(right_id, {}).get(graph_type), f"right {graph_type.upper()} graph{suffix}", axes[graph_row][1])
                for ax, method_id, side in [(axes[graph_row][2], left_id, "left"), (axes[graph_row][3], right_id, "right")]:
                    values = eigenvalues_for(method_id, graph_type)
                    ax.set_title(f"{side} {graph_type.upper()} eigenvalues{suffix}", fontsize=9)
                    if values.size:
                        ax.plot(np.arange(1, values.size + 1), np.sort(values), linewidth=1.1)
                        ax.grid(alpha=0.2)
                    else:
                        ax.text(0.5, 0.5, "missing", ha="center", va="center", transform=ax.transAxes)
                        ax.set_xticks([])
                        ax.set_yticks([])
            plt.tight_layout()
            plt.show()

        for _, row in render_cases_df.iterrows():
            show_misclassified_case(row)


In [ ]:
time_cols = ["graph_type", "metric", "k_eigen", "score_time_seconds", "threshold_time_seconds", "total_config_time_seconds"]
missing = [col for col in time_cols if col not in trained_df.columns]
if missing:
    display(HTML(
        "<b>Timing per graph/metric is not available in this tuning JSON.</b> "
        "Re-run <code>04_tune_pss_wasserstein.py</code>; new runs record "
        "<code>score_time_seconds</code>, <code>threshold_time_seconds</code>, and <code>total_config_time_seconds</code>."
    ))
else:
    timing_df = trained_df[time_cols].copy()
    timing_df["k_eigen"] = timing_df["k_eigen"].where(timing_df["k_eigen"].notna(), "full")
    display(timing_df.sort_values(["metric", "graph_type"]))

    total_by_metric = (
        timing_df.groupby("metric", as_index=False)[["score_time_seconds", "threshold_time_seconds", "total_config_time_seconds"]]
        .sum()
        .sort_values("total_config_time_seconds", ascending=False)
    )
    display(total_by_metric)

pipeline_timings = json.loads(PIPELINE_TIMINGS_PATH.read_text(encoding="utf-8")) if PIPELINE_TIMINGS_PATH.exists() else {"stages": {}}
stage = pipeline_timings.get("stages", {}).get("04_tune_pss_wasserstein", {})
if stage:
    display(pd.DataFrame([{
        "stage": "04_tune_pss_wasserstein",
        "seconds": round(stage.get("seconds", 0), 2),
        "minutes": round(stage.get("minutes", stage.get("seconds", 0) / 60), 2),
        "updated_at_utc": stage.get("updated_at_utc"),
        "source": str(PIPELINE_TIMINGS_PATH),
    }]))
